# SWOT Tidal Residual Analysis — Canal Baker

Loads SWOT WSE parquet from S3, computes FES2022 and SHOA+offset tidal
predictions, and characterizes tidal signal across the Canal Baker network.

**Prerequisites:**
- `bahia_orange_predictions.csv` in project root (manually exported from SHOA predictor)
- FES2022 model files (downloaded automatically to `/scratch/fes2022/` on first run)

**Input:** `s3://esip-qhub/canal-baker-tides/swot_wse_samples.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import s3fs
import pyTMD
from pathlib import Path

INPUT_PATH = "s3://esip-qhub/canal-baker-tides/swot_wse_samples.parquet"
FES2022_DIR = Path("/scratch/fes2022")
SHOA_CSV = Path("bahia_orange_predictions.csv")

# Time offsets (minutes) from Pub. 3009 — tide arrives this many minutes
# after Bahía Orange at each SHOA anchor port. Look these up in Pub. 3009
# for Puerto Francisco (P07) and Puerto Brown (P12) and update below.
SHOA_OFFSETS = {
    "P07": 0,   # TODO: replace with published Pub. 3009 value for Puerto Francisco
    "P12": 0,   # TODO: replace with published Pub. 3009 value for Puerto Brown
}

In [ ]:
fs = s3fs.S3FileSystem()
with fs.open(INPUT_PATH, "rb") as f:
    df = pd.read_parquet(f)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

points = pd.read_csv("sample_points.csv")

print(f"Loaded {len(df)} observations across {df['point_id'].nunique()} points")
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
display(df.head())

In [ ]:
FES2022_DIR.mkdir(parents=True, exist_ok=True)

existing = list(FES2022_DIR.glob("**/*.nc"))
if not existing:
    print("Downloading FES2022 from AVISO — requires AVISO credentials in ~/.netrc")
    print("See: https://pytmd.readthedocs.io/en/latest/getting_started/FES.html")
    import subprocess
    r = subprocess.run(
        ["python", "-m", "pyTMD.download_FES_Tidal_Models",
         "--directory", str(FES2022_DIR),
         "--model", "FES2022",
         "--tide", "ocean"],
        capture_output=True, text=True
    )
    print(r.stdout[-2000:] if r.stdout else "")
    if r.returncode != 0:
        print("DOWNLOAD ERROR:", r.stderr[-1000:])
        raise RuntimeError("FES2022 download failed — see output above and pyTMD docs")
else:
    print(f"FES2022 already present: {len(existing)} files in {FES2022_DIR}")